In [24]:
import time
import pandas as pd
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from webdriver_manager.chrome import ChromeDriverManager

# ==========================================================
# 1. MOTOR DE NAVEGACIÓN (Configuración de Laboratorio)
# ==========================================================
def iniciar_navegador():
    options = Options()
    options.binary_location = "/usr/bin/google-chrome"
    options.add_argument("--headless")
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")
    options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36")

    return webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)

# ==========================================================
# 2. CONFIGURACIÓN DEL GRUPO (Práctica con Citas)
# ==========================================================

# URL de práctica (Esta sí resuelve DNS en tu laboratorio)
URL_OBJETIVO = "https://quotes.toscrape.com/js/"

# Definir las etiquetas técnicas para este sitio específico
SELECTOR_CONTENEDOR = "div.quote"          # El bloque que contiene cada cita
SELECTOR_DATO_A = "span.text"              # La frase o texto de la cita
SELECTOR_DATO_B = "small.author"           # El nombre del autor

# ==========================================================
# 3. EJECUCIÓN DEL SCRAPING
# ==========================================================
driver = iniciar_navegador()
dataset_final = []

try:
    print(f" Conectando a la fuente de datos de prueba...")
    driver.get(URL_OBJETIVO)
    
    # Esperamos a que el JavaScript de las citas cargue el contenido
    time.sleep(5)  

    # Capturamos todos los bloques de información
    elementos = driver.find_elements(By.CSS_SELECTOR, SELECTOR_CONTENEDOR)
    print(f" Se encontraron {len(elementos)} registros potenciales.")

    for item in elementos:
        try:
            # Extracción genérica de datos
            columna_a = item.find_element(By.CSS_SELECTOR, SELECTOR_DATO_A).text
            columna_b = item.find_element(By.CSS_SELECTOR, SELECTOR_DATO_B).text

            # IMPORTANTE: Para este ejemplo de texto, quitamos el filtro de números
            # para que puedas ver el nombre del autor completo.
            valor_limpio = columna_b.strip()

            dataset_final.append({
                "cita_texto": columna_a,
                "autor": valor_limpio,
                "status": 1.0  
            })
        except:
            continue

    # ==========================================================
    # 4. SALIDA DE DATOS
    # ==========================================================
    if dataset_final:
        df = pd.DataFrame(dataset_final)
        nombre_archivo = "datos_citas_practica.csv"
        df.to_csv(nombre_archivo, index=False)
        print(f" Proceso exitoso. Archivo generado: {nombre_archivo}")
        print(df.head())
    else:
        print(" No se capturaron datos. Revisa la conexión.")

finally:
    driver.quit()

 Conectando a la fuente de datos de prueba...
 Se encontraron 10 registros potenciales.
 Proceso exitoso. Archivo generado: datos_citas_practica.csv
                                          cita_texto            autor  status
0  “The world as we have created it is a process ...  Albert Einstein     1.0
1  “It is our choices, Harry, that show what we t...     J.K. Rowling     1.0
2  “There are only two ways to live your life. On...  Albert Einstein     1.0
3  “The person, be it gentleman or lady, who has ...      Jane Austen     1.0
4  “Imperfection is beauty, madness is genius and...   Marilyn Monroe     1.0
